In [12]:
# Импорт необходимых библиотек для работы с данными
import pandas as pd
import numpy as np

скачайте датасет
https://www.kaggle.com/datasets/chiranjivdas09/ta-feng-grocery-dataset

In [2]:
!unzip ../../archive.zip 

Archive:  ../../archive.zip
  inflating: ta_feng_all_months_merged.csv  


In [14]:
# Загрузка датасета Ta Feng с покупками в продуктовом магазине
# Преобразование даты транзакций в формат datetime и сортировка по дате
df = pd.read_csv('../../ta_feng_all_months_merged.csv')
df['TRANSACTION_DT'] = pd.to_datetime(df['TRANSACTION_DT'])
df = df.sort_values(by='TRANSACTION_DT')
df.head()

,TRANSACTION_DT,CUSTOMER_ID,AGE_GROUP,PIN_CODE,PRODUCT_SUBCLASS,PRODUCT_ID,AMOUNT,ASSET,SALES_PRICE
0,2000-11-01,1104905,45-49,115,110411,4710199010372,2,24,30
907,2000-11-01,2015071,35-39,221,500509,20515461,1,18,24
906,2000-11-01,2090375,NaN,Unknown,110411,4710110241014,1,31,38
905,2000-11-01,166324,60-64,115,100505,4710154015206,1,32,37
904,2000-11-01,2015071,35-39,221,110404,4710175566374,1,88,122


In [16]:
# Переименование столбцов в стандартный формат для рекомендательных систем
# CUSTOMER_ID -> user_id, PRODUCT_ID -> item_id, TRANSACTION_DT -> timestamp
df.rename({'CUSTOMER_ID':'user_id','PRODUCT_ID':'item_id','TRANSACTION_DT':'timestamp'}, axis=1, inplace=True)
df.head()

,timestamp,user_id,AGE_GROUP,PIN_CODE,PRODUCT_SUBCLASS,item_id,AMOUNT,ASSET,SALES_PRICE
0,2000-11-01,1104905,45-49,115,110411,4710199010372,2,24,30
907,2000-11-01,2015071,35-39,221,500509,20515461,1,18,24
906,2000-11-01,2090375,NaN,Unknown,110411,4710110241014,1,31,38
905,2000-11-01,166324,60-64,115,100505,4710154015206,1,32,37
904,2000-11-01,2015071,35-39,221,110404,4710175566374,1,88,122


# Анализ повторных покупок

Посчитайте, сколько раз встречаются повторные покупки одинаковых товаров одними и теми же пользователями

In [17]:
# Подсчёт количества повторных покупок каждой пары (пользователь, товар)
# Показываем топ-30 наиболее часто повторяющихся покупок
df[['user_id','item_id']].value_counts()[:30]

user_id  item_id      
2019604  4710088433305    49
1113952  4710085120161    44
2019604  4710088433312    40
1657982  4710088433312    38
2019604  4710088434104    37
         4710105051321    28
1991741  4710088433312    26
2000558  4710058278059    25
1075274  4710094097768    24
1657982  4710088432476    23
1136791  4711080010112    22
991407   4710105015118    21
1657982  4710105051642    21
920308   4711022100017    20
1846904  4714981010038    20
1769081  4710126184268    18
309448   4710046011101    18
950565   4710105010144    17
1638080  4710105015118    17
1573879  4972045758337    17
906456   20415723         17
49122    4710105010649    17
616195   4710339000010    16
882682   4710110222228    16
427159   4711080010112    16
882682   4710110222235    16
2068718  4710085104130    16
1098143  4710421090059    15
2154978  80135906         15
2032207  4711080010112    15
Name: count, dtype: int64

## Давайте создадим leave-one-out basket

In [18]:
# Создание идентификатора корзины (basket_id) для каждого пользователя
# basket_id=1 - самая последняя покупка (последняя по времени)
# basket_id=2 - предпоследняя покупка и т.д.
# Это нужно для создания leave-one-out разбиения (последняя корзина - тест, остальные - обучение)
df['basket_id'] = df.groupby('user_id')['timestamp'].rank(method='dense', ascending=False)

# Пример для пользователя 1637335: видим его корзины от 6 (самая старая) до 1 (самая новая)
df.loc[df.user_id==1637335][['user_id','item_id','timestamp','basket_id']]

,user_id,item_id,timestamp,basket_id
6751,1637335,4712815116246,2000-11-02,6.0
6793,1637335,4710094001758,2000-11-02,6.0
6773,1637335,4711258001256,2000-11-02,6.0
5948,1637335,4710049000386,2000-11-02,6.0
5931,1637335,4901422038939,2000-11-02,6.0
9188,1637335,4710094021572,2000-11-02,6.0
7815,1637335,50000301829,2000-11-02,6.0
7723,1637335,4710363431002,2000-11-02,6.0
7951,1637335,4901872810543,2000-11-02,6.0
7539,1637335,4710101203144,2000-11-02,6.0


## Закодируйте айтемы к 0, ..., n_items

In [19]:
df.head()
# 0 ... n_users
# 0 ... n_items

,timestamp,user_id,AGE_GROUP,PIN_CODE,PRODUCT_SUBCLASS,item_id,AMOUNT,ASSET,SALES_PRICE,basket_id
0,2000-11-01,1104905,45-49,115,110411,4710199010372,2,24,30,13.0
907,2000-11-01,2015071,35-39,221,500509,20515461,1,18,24,9.0
906,2000-11-01,2090375,NaN,Unknown,110411,4710110241014,1,31,38,24.0
905,2000-11-01,166324,60-64,115,100505,4710154015206,1,32,37,17.0
904,2000-11-01,2015071,35-39,221,110404,4710175566374,1,88,122,9.0


In [20]:
# Кодирование пользователей и товаров в последовательные числовые идентификаторы
# user_id: 0, 1, 2, ..., n_users-1
# item_id: 0, 1, 2, ..., n_items-1
# Это необходимо для корректной работы алгоритмов рекомендаций
user2id = {x: idx for idx, x in enumerate(df.user_id.unique())}
item2id = {x: idx for idx, x in enumerate(df.item_id.unique())}

df['user_id'] = df['user_id'].apply(lambda x: user2id[x])
df['item_id'] = df['item_id'].apply(lambda x: item2id[x])

In [21]:
df.head()

,timestamp,user_id,AGE_GROUP,PIN_CODE,PRODUCT_SUBCLASS,item_id,AMOUNT,ASSET,SALES_PRICE,basket_id
0,2000-11-01,0,45-49,115,110411,0,2,24,30,13.0
907,2000-11-01,1,35-39,221,500509,1,1,18,24,9.0
906,2000-11-01,2,NaN,Unknown,110411,2,1,31,38,24.0
905,2000-11-01,3,60-64,115,100505,3,1,32,37,17.0
904,2000-11-01,1,35-39,221,110404,4,1,88,122,9.0


In [22]:
# Разделение данных на обучающую и тестовую выборки по принципу leave-one-out
# test_baskets: последняя корзина каждого пользователя (basket_id=1) - это то, что нужно предсказать
# train_baskets: все остальные корзины (basket_id!=1) - для обучения моделей
test_baskets = df.loc[df.basket_id==1].copy()
train_baskets = df.loc[df.basket_id!=1].copy()

train_baskets.head()

,timestamp,user_id,AGE_GROUP,PIN_CODE,PRODUCT_SUBCLASS,item_id,AMOUNT,ASSET,SALES_PRICE,basket_id
0,2000-11-01,0,45-49,115,110411,0,2,24,30,13.0
907,2000-11-01,1,35-39,221,500509,1,1,18,24,9.0
906,2000-11-01,2,NaN,Unknown,110411,2,1,31,38,24.0
905,2000-11-01,3,60-64,115,100505,3,1,32,37,17.0
904,2000-11-01,1,35-39,221,110404,4,1,88,122,9.0


In [23]:
# Группировка данных по пользователям
# Для каждого пользователя создаём список пар (basket_id, item_id)
# train_grouped: история покупок для обучения
# test_grouped: последняя корзина для тестирования
train_grouped = train_baskets.groupby('user_id').apply(lambda x: list(zip(x.basket_id, x.item_id))).reset_index()
train_grouped.rename({0: 'train_baskets'}, axis=1, inplace=True)

test_grouped = test_baskets.groupby('user_id').apply(lambda x: list(zip(x.basket_id, x.item_id))).reset_index()
test_grouped.rename({0: 'test_baskets'}, axis=1, inplace=True)

/var/folders/gw/9vsxt6xx1t39l8d2vg28flr00000gq/T/ipykernel_66610/250037549.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  train_grouped = train_baskets.groupby('user_id').apply(lambda x: list(zip(x.basket_id, x.item_id))).reset_index()
/var/folders/gw/9vsxt6xx1t39l8d2vg28flr00000gq/T/ipykernel_66610/250037549.py:8: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  test_grouped = test_baskets.groupby('user_

In [24]:
train_grouped.head()

,user_id,train_baskets
0,0,"[(13.0, 0), (13.0, 453), (13.0, 223), (13.0, 5..."
1,1,"[(9.0, 1), (9.0, 4), (9.0, 17), (9.0, 25), (9...."
2,2,"[(24.0, 2), (24.0, 8), (24.0, 15), (24.0, 20),..."
3,3,"[(17.0, 3), (17.0, 7), (17.0, 11), (17.0, 13),..."
4,4,"[(33.0, 5), (33.0, 30), (33.0, 46), (33.0, 58)..."


In [25]:
test_grouped.head()

,user_id,test_baskets
0,0,"[(1.0, 60), (1.0, 3152), (1.0, 267), (1.0, 861..."
1,1,"[(1.0, 216), (1.0, 5029), (1.0, 865), (1.0, 23..."
2,2,"[(1.0, 1603)]"
3,3,"[(1.0, 3012), (1.0, 13), (1.0, 883), (1.0, 3712)]"
4,4,"[(1.0, 1011), (1.0, 6347), (1.0, 282), (1.0, 9..."


In [26]:
# Объединение обучающих и тестовых данных в один DataFrame
# Каждая строка содержит user_id, историю покупок (train_baskets) и тестовую корзину (test_baskets)
grouped = train_grouped.merge(test_grouped)
grouped.head()

,user_id,train_baskets,test_baskets
0,0,"[(13.0, 0), (13.0, 453), (13.0, 223), (13.0, 5...","[(1.0, 60), (1.0, 3152), (1.0, 267), (1.0, 861..."
1,1,"[(9.0, 1), (9.0, 4), (9.0, 17), (9.0, 25), (9....","[(1.0, 216), (1.0, 5029), (1.0, 865), (1.0, 23..."
2,2,"[(24.0, 2), (24.0, 8), (24.0, 15), (24.0, 20),...","[(1.0, 1603)]"
3,3,"[(17.0, 3), (17.0, 7), (17.0, 11), (17.0, 13),...","[(1.0, 3012), (1.0, 13), (1.0, 883), (1.0, 3712)]"
4,4,"[(33.0, 5), (33.0, 30), (33.0, 46), (33.0, 58)...","[(1.0, 1011), (1.0, 6347), (1.0, 282), (1.0, 9..."


будем делать рекомендации на 20 элементах

In [27]:
# Количество рекомендаций для генерации (top-20)
k = 20

реализуйте 20 рандомный рекомендаций

In [32]:
# Базовая модель: случайные рекомендации (Random Baseline)
# Выбираем 20 случайных товаров из всего ассортимента
random_recs = np.random.choice(np.arange(len(item2id)), size=k)

In [33]:
random_recs[None,:]

array([[ 2338,  2042, 11087,  4728, 21899, 19204,  7758, 10660,  6396,
         9609, 14001,  8419,  1215,  9078, 16482, 21201, 14459,  2752,
        12880, 10462]])

In [34]:
# Добавляем случайные рекомендации для всех пользователей (одинаковые для всех)
grouped['random_recs'] = grouped.user_id.apply(lambda x: random_recs)
grouped.head()

,user_id,train_baskets,test_baskets,random_recs
0,0,"[(13.0, 0), (13.0, 453), (13.0, 223), (13.0, 5...","[(1.0, 60), (1.0, 3152), (1.0, 267), (1.0, 861...","[2338, 2042, 11087, 4728, 21899, 19204, 7758, ..."
1,1,"[(9.0, 1), (9.0, 4), (9.0, 17), (9.0, 25), (9....","[(1.0, 216), (1.0, 5029), (1.0, 865), (1.0, 23...","[2338, 2042, 11087, 4728, 21899, 19204, 7758, ..."
2,2,"[(24.0, 2), (24.0, 8), (24.0, 15), (24.0, 20),...","[(1.0, 1603)]","[2338, 2042, 11087, 4728, 21899, 19204, 7758, ..."
3,3,"[(17.0, 3), (17.0, 7), (17.0, 11), (17.0, 13),...","[(1.0, 3012), (1.0, 13), (1.0, 883), (1.0, 3712)]","[2338, 2042, 11087, 4728, 21899, 19204, 7758, ..."
4,4,"[(33.0, 5), (33.0, 30), (33.0, 46), (33.0, 58)...","[(1.0, 1011), (1.0, 6347), (1.0, 282), (1.0, 9...","[2338, 2042, 11087, 4728, 21899, 19204, 7758, ..."


In [35]:
# Функции для оценки качества рекомендаций

def ndcg_metric(gt_items, predicted):
    """
    Normalized Discounted Cumulative Gain (NDCG)
    Метрика, учитывающая порядок рекомендаций: чем выше релевантный товар в списке, тем лучше
    """
    at = len(predicted)
    relevance = np.array([1 if x in predicted else 0 for x in gt_items])
    # DCG uses the relevance of the recommended items
    rank_dcg = dcg(relevance)

    if rank_dcg == 0.0:
        return 0.0

    # IDCG has all relevances to 1 (or the values provided), up to the number of items in the test set that can fit in the list length
    ideal_dcg = dcg(np.sort(relevance)[::-1][:at])

    if ideal_dcg == 0.0:
        return 0.0

    ndcg_ = rank_dcg / ideal_dcg

    return ndcg_


def dcg(scores):
    """Discounted Cumulative Gain - вспомогательная функция для расчёта NDCG"""
    return np.sum(np.divide(np.power(2, scores) - 1, np.log2(np.arange(scores.shape[0], dtype=np.float64) + 2)),
                  dtype=np.float64)


def recall_metric(gt_items, predicted):
    """
    Recall@k (полнота)
    Доля товаров из тестовой корзины, которые попали в топ-k рекомендаций
    """
    n_gt = len(gt_items)
    intersection = len(set(gt_items).intersection(set(predicted)))
    return intersection / n_gt

def evaluate_recommender(df, model_preds, gt_col='test_interactions', topn=10):
    """
    Оценка качества рекомендательной системы
    Возвращает средние значения NDCG и Recall по всем пользователям
    """
    metric_values = []
    
    for idx, row in df.iterrows():
        gt_items = [x[1] for x in row[gt_col]]  # Извлекаем item_id из тестовой корзины
        metric_values.append((ndcg_metric(gt_items, row[model_preds]),
                              recall_metric(gt_items, row[model_preds])))
        
    return {'ndcg':np.mean([x[0] for x in metric_values]),
            'recall':np.mean([x[1] for x in metric_values])}


In [36]:
# Оценка качества случайных рекомендаций (Random Baseline)
# Результаты очень низкие, так как случайные рекомендации почти никогда не попадают в тестовую корзину
evaluate_recommender(grouped, gt_col='test_baskets', model_preds='random_recs', topn=k)

{'ndcg': 0.001416583586408769, 'recall': 0.00037208968792657403}

In [37]:
len(item2id)

23812

## Теперь можем строить рекомендации

### Постройте рекомендации по популярности по количеству покупок и оцените качество

In [38]:
train_baskets.item_id.value_counts()[:30]

item_id
13       5494
6        4614
13700    1809
57       1712
3639     1570
509      1508
290      1372
1761     1368
9        1289
62       1287
407      1284
1302     1278
60       1202
956      1187
173      1164
69       1163
50       1080
1956     1008
14        988
4914      932
3543      903
1368      902
1455      900
1763      879
1345      818
154       812
283       806
2282      790
55        789
2358      785
Name: count, dtype: int64

In [40]:
# Модель популярности #1: топ-20 самых часто покупаемых товаров (по количеству покупок)
# Считаем общее количество покупок каждого товара во всех корзинах
most_pop = train_baskets.item_id.value_counts().index.tolist()[:k]
most_pop

[13,
 6,
 13700,
 57,
 3639,
 509,
 290,
 1761,
 9,
 62,
 407,
 1302,
 60,
 956,
 173,
 69,
 50,
 1956,
 14,
 4914]

In [41]:
grouped['mostpoprecs'] = grouped.user_id.apply(lambda x: most_pop)
grouped.head()

,user_id,train_baskets,test_baskets,random_recs,mostpoprecs
0,0,"[(13.0, 0), (13.0, 453), (13.0, 223), (13.0, 5...","[(1.0, 60), (1.0, 3152), (1.0, 267), (1.0, 861...","[2338, 2042, 11087, 4728, 21899, 19204, 7758, ...","[13, 6, 13700, 57, 3639, 509, 290, 1761, 9, 62..."
1,1,"[(9.0, 1), (9.0, 4), (9.0, 17), (9.0, 25), (9....","[(1.0, 216), (1.0, 5029), (1.0, 865), (1.0, 23...","[2338, 2042, 11087, 4728, 21899, 19204, 7758, ...","[13, 6, 13700, 57, 3639, 509, 290, 1761, 9, 62..."
2,2,"[(24.0, 2), (24.0, 8), (24.0, 15), (24.0, 20),...","[(1.0, 1603)]","[2338, 2042, 11087, 4728, 21899, 19204, 7758, ...","[13, 6, 13700, 57, 3639, 509, 290, 1761, 9, 62..."
3,3,"[(17.0, 3), (17.0, 7), (17.0, 11), (17.0, 13),...","[(1.0, 3012), (1.0, 13), (1.0, 883), (1.0, 3712)]","[2338, 2042, 11087, 4728, 21899, 19204, 7758, ...","[13, 6, 13700, 57, 3639, 509, 290, 1761, 9, 62..."
4,4,"[(33.0, 5), (33.0, 30), (33.0, 46), (33.0, 58)...","[(1.0, 1011), (1.0, 6347), (1.0, 282), (1.0, 9...","[2338, 2042, 11087, 4728, 21899, 19204, 7758, ...","[13, 6, 13700, 57, 3639, 509, 290, 1761, 9, 62..."


In [42]:
# Оценка качества рекомендаций по популярности #1
# NDCG ~0.19, Recall ~0.097 - значительно лучше случайных рекомендаций
evaluate_recommender(grouped, gt_col='test_baskets',model_preds='mostpoprecs', topn=k)

{'ndcg': 0.1902223218557026, 'recall': 0.0969647255458864}

## Популярность по уникальным клиентам

Теперь посчитайте популярность по количеству уникальных пользователей, которые купили товар (а не по общему количеству покупок)

In [43]:
mostpop2 = train_baskets.drop_duplicates(subset=['user_id','item_id']).item_id.value_counts().index.tolist()[:k]
mostpop2

[6,
 13,
 13700,
 3639,
 1761,
 509,
 407,
 57,
 1302,
 290,
 9,
 956,
 60,
 50,
 62,
 69,
 14,
 173,
 1368,
 3543]

In [28]:
most_pop

[13,
 6,
 13700,
 57,
 3639,
 509,
 290,
 1761,
 9,
 62,
 407,
 1302,
 60,
 956,
 173,
 69,
 50,
 1956,
 14,
 4914]

In [44]:
grouped['mostpoprecs2'] = grouped.user_id.apply(lambda x: mostpop2)
grouped.head()

,user_id,train_baskets,test_baskets,random_recs,mostpoprecs,mostpoprecs2
0,0,"[(13.0, 0), (13.0, 453), (13.0, 223), (13.0, 5...","[(1.0, 60), (1.0, 3152), (1.0, 267), (1.0, 861...","[2338, 2042, 11087, 4728, 21899, 19204, 7758, ...","[13, 6, 13700, 57, 3639, 509, 290, 1761, 9, 62...","[6, 13, 13700, 3639, 1761, 509, 407, 57, 1302,..."
1,1,"[(9.0, 1), (9.0, 4), (9.0, 17), (9.0, 25), (9....","[(1.0, 216), (1.0, 5029), (1.0, 865), (1.0, 23...","[2338, 2042, 11087, 4728, 21899, 19204, 7758, ...","[13, 6, 13700, 57, 3639, 509, 290, 1761, 9, 62...","[6, 13, 13700, 3639, 1761, 509, 407, 57, 1302,..."
2,2,"[(24.0, 2), (24.0, 8), (24.0, 15), (24.0, 20),...","[(1.0, 1603)]","[2338, 2042, 11087, 4728, 21899, 19204, 7758, ...","[13, 6, 13700, 57, 3639, 509, 290, 1761, 9, 62...","[6, 13, 13700, 3639, 1761, 509, 407, 57, 1302,..."
3,3,"[(17.0, 3), (17.0, 7), (17.0, 11), (17.0, 13),...","[(1.0, 3012), (1.0, 13), (1.0, 883), (1.0, 3712)]","[2338, 2042, 11087, 4728, 21899, 19204, 7758, ...","[13, 6, 13700, 57, 3639, 509, 290, 1761, 9, 62...","[6, 13, 13700, 3639, 1761, 509, 407, 57, 1302,..."
4,4,"[(33.0, 5), (33.0, 30), (33.0, 46), (33.0, 58)...","[(1.0, 1011), (1.0, 6347), (1.0, 282), (1.0, 9...","[2338, 2042, 11087, 4728, 21899, 19204, 7758, ...","[13, 6, 13700, 57, 3639, 509, 290, 1761, 9, 62...","[6, 13, 13700, 3639, 1761, 509, 407, 57, 1302,..."


In [45]:
# Оценка качества рекомендаций по популярности #2 (по уникальным пользователям)
# NDCG ~0.197, Recall ~0.102 - немного лучше, чем популярность по количеству покупок
evaluate_recommender(grouped, gt_col='test_baskets',model_preds='mostpoprecs2', topn=k)

{'ndcg': 0.19671364391441537, 'recall': 0.10134244721016634}

In [46]:
evaluate_recommender(grouped, gt_col='test_baskets',model_preds='mostpoprecs', topn=k)

{'ndcg': 0.1902223218557026, 'recall': 0.0969647255458864}

## Сравните рекомендации между двумя видами популярности

In [32]:
# Здесь можно добавить сравнение двух подходов к популярности:
# 1. most_pop - по общему количеству покупок (один товар может быть куплен одним пользователем много раз)
# 2. mostpop2 - по количеству уникальных пользователей, купивших товар

In [47]:
# Анализ распределения тестовых покупок по датам
# Помогает понять сезонность и временные паттерны покупок
test_baskets.timestamp.value_counts()[:30]

timestamp
2001-02-25    11673
2001-02-28     9375
2001-02-24     8183
2001-01-21     7634
2001-01-22     6833
2001-02-11     6625
2001-02-18     6348
2001-02-10     5424
2001-02-04     5313
2001-02-17     4811
2001-02-23     4553
2001-02-03     4444
2001-02-27     4432
2001-02-21     4367
2001-01-14     4313
2001-02-22     4156
2001-02-26     4154
2001-02-19     4140
2001-01-20     3984
2001-02-20     3640
2000-12-17     3467
2001-02-15     3398
2000-12-31     3042
2001-02-16     3020
2001-01-01     3020
2001-02-14     3010
2001-01-23     2897
2001-01-13     2867
2001-01-18     2816
2001-01-19     2669
Name: count, dtype: int64

## Давайте сделаем рекомендации по персональной популярности

In [48]:
def count_personal_recs(row):
    """
    Персонализированные рекомендации на основе истории покупок пользователя
    Рекомендуем топ-20 товаров, которые пользователь покупал чаще всего
    """
    counts = {}
    for l in row.train_baskets:
        item = l[1]
        if item in counts:
            counts[item] += 1
        else:
            counts[item] = 1
    
    # Сортируем товары по частоте покупок пользователем
    counts = list(sorted(counts.items(), key=lambda x: -x[1]))
    items = [x[0] for x in counts] 
    
    return items[:k]


# Генерируем персонализированные рекомендации для каждого пользователя
grouped['user_popular'] = grouped.apply(lambda x: count_personal_recs(x), axis=1)
grouped.head()

,user_id,train_baskets,test_baskets,random_recs,mostpoprecs,mostpoprecs2,user_popular
0,0,"[(13.0, 0), (13.0, 453), (13.0, 223), (13.0, 5...","[(1.0, 60), (1.0, 3152), (1.0, 267), (1.0, 861...","[2338, 2042, 11087, 4728, 21899, 19204, 7758, ...","[13, 6, 13700, 57, 3639, 509, 290, 1761, 9, 62...","[6, 13, 13700, 3639, 1761, 509, 407, 57, 1302,...","[453, 60, 15042, 1060, 1956, 12913, 83, 8615, ..."
1,1,"[(9.0, 1), (9.0, 4), (9.0, 17), (9.0, 25), (9....","[(1.0, 216), (1.0, 5029), (1.0, 865), (1.0, 23...","[2338, 2042, 11087, 4728, 21899, 19204, 7758, ...","[13, 6, 13700, 57, 3639, 509, 290, 1761, 9, 62...","[6, 13, 13700, 3639, 1761, 509, 407, 57, 1302,...","[3620, 400, 13, 4624, 1405, 1, 4, 17, 25, 26, ..."
2,2,"[(24.0, 2), (24.0, 8), (24.0, 15), (24.0, 20),...","[(1.0, 1603)]","[2338, 2042, 11087, 4728, 21899, 19204, 7758, ...","[13, 6, 13700, 57, 3639, 509, 290, 1761, 9, 62...","[6, 13, 13700, 3639, 1761, 509, 407, 57, 1302,...","[2396, 3710, 8, 306, 6009, 611, 1653, 8031, 13..."
3,3,"[(17.0, 3), (17.0, 7), (17.0, 11), (17.0, 13),...","[(1.0, 3012), (1.0, 13), (1.0, 883), (1.0, 3712)]","[2338, 2042, 11087, 4728, 21899, 19204, 7758, ...","[13, 6, 13700, 57, 3639, 509, 290, 1761, 9, 62...","[6, 13, 13700, 3639, 1761, 509, 407, 57, 1302,...","[55, 6, 3, 7, 11, 13, 24, 27, 29, 31, 39, 40, ..."
4,4,"[(33.0, 5), (33.0, 30), (33.0, 46), (33.0, 58)...","[(1.0, 1011), (1.0, 6347), (1.0, 282), (1.0, 9...","[2338, 2042, 11087, 4728, 21899, 19204, 7758, ...","[13, 6, 13700, 57, 3639, 509, 290, 1761, 9, 62...","[6, 13, 13700, 3639, 1761, 509, 407, 57, 1302,...","[792, 282, 790, 2309, 5871, 1883, 7750, 58, 68..."


In [49]:
# Оценка персонализированных рекомендаций
# NDCG ~0.232, Recall ~0.114 - заметно лучше глобальной популярности!
# Персонализация помогает: учёт индивидуальных предпочтений пользователя даёт прирост качества
evaluate_recommender(grouped, gt_col='test_baskets',model_preds='user_popular', topn=k)

{'ndcg': 0.23166610618797306, 'recall': 0.11365481405632985}

In [50]:
{'ndcg': 0.19671364391441537, 'recall': 0.10134244721016634}

{'ndcg': 0.19671364391441537, 'recall': 0.10134244721016634}

## Гибридная модель TopPersonal

Реализуем эвристику, которая комбинирует два сигнала:
- **f(u, i)** - персональная частота покупок товара пользователем
- **p(i)** - глобальная популярность товара

Формула: **s(u, i) = a × f(u, i) + (1-a) × p(i)**

Это помогает балансировать между персонализацией и популярностью

In [52]:
# s = sorted(scores, lambda x: (freq, popularity))

# (batch, num_items)
# freq + popularity 
# popularity > 1 -> (0, 1)

popularity = train_baskets.drop_duplicates(subset=['user_id','item_id']).item_id.value_counts().reset_index()
popularity['count'] /= popularity['count'].max()
populartiy = {k: v for k, v in zip(popularity.item_id, popularity['count'])}


In [38]:
# Нормализация популярности товаров в диапазон [0, 1]
# Это необходимо для корректного взвешивания в гибридной модели

In [53]:
def count_personal_recs(row):
    """
    Гибридные рекомендации: комбинация персональной истории и глобальной популярности
    Учитываем как частоту покупок пользователя, так и общую популярность товара
    """
    counts = {}
    for l in row.train_baskets:
        item = l[1]
        if item in counts:
            counts[item] += 1
        else:
            counts[item] = 1
    
    # Добавляем глобальную популярность к персональной частоте
    for t in counts:
        counts[t] += popularity.get(t, 0)
    
    counts = list(sorted(counts.items(), key=lambda x: -x[1]))
    items = [x[0] for x in counts][:k] 
    
    # Если товаров меньше k, дополняем глобально популярными товарами
    items = items + [x for x in mostpop2 if x not in items]
    
    return items[:k]


# Генерируем гибридные рекомендации
grouped['toppersonal'] = grouped.apply(lambda x: count_personal_recs(x), axis=1)
grouped.head()

,user_id,train_baskets,test_baskets,random_recs,mostpoprecs,mostpoprecs2,user_popular,toppersonal
0,0,"[(13.0, 0), (13.0, 453), (13.0, 223), (13.0, 5...","[(1.0, 60), (1.0, 3152), (1.0, 267), (1.0, 861...","[2338, 2042, 11087, 4728, 21899, 19204, 7758, ...","[13, 6, 13700, 57, 3639, 509, 290, 1761, 9, 62...","[6, 13, 13700, 3639, 1761, 509, 407, 57, 1302,...","[453, 60, 15042, 1060, 1956, 12913, 83, 8615, ...","[453, 60, 15042, 1060, 1956, 12913, 83, 8615, ..."
1,1,"[(9.0, 1), (9.0, 4), (9.0, 17), (9.0, 25), (9....","[(1.0, 216), (1.0, 5029), (1.0, 865), (1.0, 23...","[2338, 2042, 11087, 4728, 21899, 19204, 7758, ...","[13, 6, 13700, 57, 3639, 509, 290, 1761, 9, 62...","[6, 13, 13700, 3639, 1761, 509, 407, 57, 1302,...","[3620, 400, 13, 4624, 1405, 1, 4, 17, 25, 26, ...","[3620, 400, 13, 4624, 1405, 1, 4, 17, 25, 26, ..."
2,2,"[(24.0, 2), (24.0, 8), (24.0, 15), (24.0, 20),...","[(1.0, 1603)]","[2338, 2042, 11087, 4728, 21899, 19204, 7758, ...","[13, 6, 13700, 57, 3639, 509, 290, 1761, 9, 62...","[6, 13, 13700, 3639, 1761, 509, 407, 57, 1302,...","[2396, 3710, 8, 306, 6009, 611, 1653, 8031, 13...","[2396, 3710, 8, 306, 6009, 611, 1653, 8031, 13..."
3,3,"[(17.0, 3), (17.0, 7), (17.0, 11), (17.0, 13),...","[(1.0, 3012), (1.0, 13), (1.0, 883), (1.0, 3712)]","[2338, 2042, 11087, 4728, 21899, 19204, 7758, ...","[13, 6, 13700, 57, 3639, 509, 290, 1761, 9, 62...","[6, 13, 13700, 3639, 1761, 509, 407, 57, 1302,...","[55, 6, 3, 7, 11, 13, 24, 27, 29, 31, 39, 40, ...","[55, 6, 3, 7, 11, 13, 24, 27, 29, 31, 39, 40, ..."
4,4,"[(33.0, 5), (33.0, 30), (33.0, 46), (33.0, 58)...","[(1.0, 1011), (1.0, 6347), (1.0, 282), (1.0, 9...","[2338, 2042, 11087, 4728, 21899, 19204, 7758, ...","[13, 6, 13700, 57, 3639, 509, 290, 1761, 9, 62...","[6, 13, 13700, 3639, 1761, 509, 407, 57, 1302,...","[792, 282, 790, 2309, 5871, 1883, 7750, 58, 68...","[792, 282, 790, 2309, 5871, 1883, 7750, 58, 68..."


In [54]:
# Оценка гибридной модели TopPersonal
# NDCG ~0.290, Recall ~0.152 - лучший результат среди всех моделей!
# Комбинация персонализации и популярности даёт наилучшее качество
evaluate_recommender(grouped, gt_col='test_baskets',model_preds='toppersonal', topn=k)

{'ndcg': 0.29006386845146764, 'recall': 0.1524134873788588}

In [55]:
evaluate_recommender(grouped, gt_col='test_baskets',model_preds='mostpoprecs', topn=k)

{'ndcg': 0.1902223218557026, 'recall': 0.0969647255458864}

## Рекомендации по карточке


In [56]:
corpus = [
    "Молоко",
    "Хлеб",
    "Яйца",
    "Масло сливочное",
    "Масло подсолнечное",
    "Сахар",
    "Соль",
    "Мука",
    "Рис",
    "Макаронные изделия",
    "Консервированные овощи (кукуруза, горошек, фасоль)",
    "Консервированные фрукты (ананасы, персики)",
    "Кетчуп",
    "Майонез",
    "Горчица",
    "Чай",
    "Кофе",
    "Соки (апельсиновый, яблочный, мультифрукт)",
    "Газированные напитки (кола, лимонад)",
    "Минеральная вода",
    "Йогурты",
    "Сыр",
    "Колбаса",
    "Сосиски",
    "Замороженные овощи",
    "Замороженные полуфабрикаты (пельмени, котлеты)",
    "Чипсы",
    "Печенье",
    "Шоколад",
    "Конфеты",
    "Мороженое",
    "Средства для мытья посуды",
    "Стиральный порошок",
    "Мыло",
    "Шампунь",
    "Зубная паста",
    "Туалетная бумага",
    "Бумажные полотенца",
    "Влажные салфетки",
    "Кошачий корм",
    "Собачий корм",
    "Детское питание",
    "Подгузники",
    "Специи (перец, лавровый лист, корица)",
    "Орехи и сухофрукты",
    "Мед",
    "Джемы и варенье",
    "Соевый соус",
    "Оливки и маслины",
    "Снеки (орешки, сухарики)",
    "Энергетические батончики",
    "Творог",
    "Сметана",
    "Кефир",
    "Ряженка",
    "Сгущенное молоко",
    "Маргарин",
    "Крупы (гречка, овсянка, пшено)",
    "Мюсли",
    "Хлопья кукурузные",
    "Сухие завтраки",
    "Консервы рыбные (тунец, сайра, шпроты)",
    "Консервы мясные (тушенка)",
    "Паштеты",
    "Соусы (сырный, барбекю, тартар)",
    "Уксус",
    "Крахмал",
    "Разрыхлитель теста",
    "Дрожжи",
    "Ванильный сахар",
    "Какао-порошок",
    "Сгущенка вареная",
    "Мармелад",
    "Зефир",
    "Пастила",
    "Халва",
    "Семечки подсолнечные",
    "Семечки тыквенные",
    "Крекеры",
    "Гранола",
    "Сухие супы",
    "Бульонные кубики",
    "Лапша быстрого приготовления",
    "Картофельное пюре быстрого приготовления",
    "Соевое молоко",
    "Миндальное молоко",
    "Кокосовое молоко",
    "Овсяное молоко",
    "Протеиновые батончики",
    "Протеиновые коктейли",
    "Детские каши",
    "Детские пюре",
    "Детские соки",
    "Детские печенья",
    "Детские молочные смеси",
    "Детские влажные салфетки",
    "Детские шампуни",
    "Детские кремы",
    "Детские зубные пасты",
    "Детские игрушки для ванной",
    "Детские бутылочки",
    "Детские соски",
    "Детские подгузники-трусики",
    "Детские влажные салфетки с кремом",
    "Детские масла для тела",
    "Детские присыпки",
    "Детские солнцезащитные кремы",
    "Детские зубные щетки"
]





In [6]:
#!pip3 install sentence_transformers

In [57]:
from sentence_transformers import SentenceTransformer, util
import torch
import numpy as np
from scipy.spatial.distance import cosine

embedder = SentenceTransformer('../../fred_t5_ru_turbo_alpaca')

No sentence-transformers model found with name ../../fred_t5_ru_turbo_alpaca. Creating a new one with mean pooling.
Loading weights: 100%|██████████| 220/220 [00:00<00:00, 1832.02it/s, Materializing param=shared.weight]                                                     
The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
T5EncoderModel LOAD REPORT from: ../../fred_t5_ru_turbo_alpaca
Key            | Status     |  | 
---------------+------------+--+-
lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [58]:
embedder

SentenceTransformer(
  (0): Transformer({'max_seq_length': None, 'do_lower_case': False, 'architecture': 'T5EncoderModel'})
  (1): Pooling({'word_embedding_dimension': 1536, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
)

In [59]:
%%time

corpus_embeddings = embedder.encode(corpus, convert_to_tensor=True)

CPU times: user 125 ms, sys: 979 ms, total: 1.1 s
Wall time: 3.67 s


In [60]:
corpus_embeddings

tensor([[-0.0020, -0.0041, -0.0032,  ...,  0.0020,  0.0032,  0.0021],
        [-0.0030, -0.0036,  0.0036,  ..., -0.0011, -0.0009, -0.0007],
        [-0.0070,  0.0063, -0.0007,  ...,  0.0029, -0.0010,  0.0024],
        ...,
        [-0.0071,  0.0083,  0.0030,  ...,  0.0042,  0.0006, -0.0015],
        [-0.0033,  0.0162,  0.0006,  ...,  0.0044,  0.0016, -0.0015],
        [-0.0005,  0.0082, -0.0018,  ...,  0.0012, -0.0021, -0.0004]],
       device='mps:0', dtype=torch.bfloat16)

In [61]:
#Sentences we want to encode. Example:
queries = np.random.choice(corpus, size=50)

counts = 0
# Find the closest 5 sentences of the corpus for each query sentence based on cosine similarity
top_k = min(10, len(corpus))

for idx, query in enumerate(queries):
    query_embedding = embedder.encode(query, convert_to_tensor=True)
    # We use cosine-similarity and torch.topk to find the highest 5 scores
    # cos_scores = util.dot_score(query_embedding, corpus_embeddings)[0]
    cos_scores = util.cos_sim(query_embedding, corpus_embeddings)[0]
    top_results = torch.topk(cos_scores, k=top_k)#.numpy()
    gpt_recs = []
    for score, idx in zip(top_results[0], top_results[1]):
        #print(f'{counts+1})', corpus[idx], "(Score: {:.4f})".format(score))
        counts += 1
        gpt_recs.append(corpus[idx][:30])

    

    print("\n\n======================\n\n")
    print("Query:", query)
    print("\nTop 10 most similar sentences in corpus:")

    counts = 0
    for idx, rec in enumerate(gpt_recs):
        print(f'{counts+1})', rec)

        counts += 1
    
#     # Alternatively, we can also use util.semantic_search to perform cosine similarty + topk
#     hits = util.semantic_search(query_embedding, corpus_embeddings, top_k=5)
#     hits = hits[0]      #Get the hits for the first query
#     for hit in hits:
#         print(corpus[hit['corpus_id']], "(Score: {:.4f})".format(hit['score']))
    






Query: Детское питание

Top 10 most similar sentences in corpus:
1) Детское питание
2) Детские соки
3) Детские зубные пасты
4) Детские соски
5) Детские игрушки для ванной
6) Детские печенья
7) Детские масла для тела
8) Детские бутылочки
9) Соевое молоко
10) Овсяное молоко




Query: Подгузники

Top 10 most similar sentences in corpus:
1) Подгузники
2) Детские подгузники-трусики
3) Детские игрушки для ванной
4) Детские влажные салфетки
5) Детские бутылочки
6) Паштеты
7) Детские зубные пасты
8) Бумажные полотенца
9) Соевое молоко
10) Детские соски




Query: Снеки (орешки, сухарики)

Top 10 most similar sentences in corpus:
1) Снеки (орешки, сухарики)
2) Орехи и сухофрукты
3) Консервированные фрукты (анана
4) Замороженные полуфабрикаты (пе
5) Соки (апельсиновый, яблочный, 
6) Газированные напитки (кола, ли
7) Крупы (гречка, овсянка, пшено)
8) Консервы рыбные (тунец, сайра,
9) Соусы (сырный, барбекю, тартар
10) Консервированные овощи (кукуру




Query: Гранола

Top 10 most similar sen